In [ ]:
import gymnasium as gym
from gymnasium.spaces import Discrete

import numpy as np
from tqdm.auto import tqdm

from dataclasses import dataclass

In [2]:
type State = int
type Action = int
type Reward = float

In [ ]:
@dataclass
class Parameters:
  alpha: float  # step size
  epsilon: float  # e-greedy
  gamma: float  # discount factor


def select_action(q_table: np.ndarray, state: State, epsilon: float, env: gym.Env) -> Action:
  # using e-greedy method

  assert isinstance(env.action_space, Discrete)
  n_actions = env.action_space.n
  
  q: np.ndarray = q_table[state]
  assert q.shape == (n_actions,)

  # exploratory
  if env.np_random.random() < epsilon:
    return env.action_space.sample()
  

  return int(env.np_random.choice(np.flatnonzero(q == q.max())))  # break ties randomly

In [4]:
def update(
  q_table: np.ndarray,
  state: State,
  new_state: State,
  action: Action,
  new_action: Action,
  reward: Reward,
  done: bool,
  params: Parameters,
):
  q_next = (1 - done) * q_table[new_state, new_action]
  td_error = reward + params.gamma * q_next - q_table[state, action]
  q_table[state, action] += params.alpha * td_error


In [5]:
def to_coord(pos: int) -> tuple[int, int]:
  x = pos % 12
  y = pos // 12

  return x, y

In [6]:
environ: gym.Env[int, int] = gym.make("CliffWalking-v1", max_episode_steps=10_000, render_mode="ansi")

In [7]:
assert isinstance(environ.action_space, Discrete)
assert isinstance(environ.observation_space, Discrete)


n_actions = int(environ.action_space.n)
obs_dim = int(environ.observation_space.n)

In [8]:
# init params
params = Parameters(alpha=0.1, epsilon=0.1, gamma=0.99)
n_episodes = 1000

In [9]:
total_rewards = []

q_table = np.zeros((obs_dim, n_actions))

pbar_ep = tqdm(range(n_episodes), desc="Episodes")
for ep in pbar_ep:
  done = False

  total_reward = 0
  observation, info = environ.reset()

  action = select_action(q_table, observation, params.epsilon, n_actions, environ)

  while not done:

    # take the action
    new_observation, reward, terminated, truncated, info = environ.step(action)

    new_action = select_action(q_table, new_observation, params.epsilon, n_actions, environ)

    done = terminated or truncated

    # update
    update(q_table, observation, new_observation, action, new_action, float(reward), done, params)

    observation = new_observation
    action = new_action

    total_reward += float(reward)

  total_rewards.append(total_reward)

  print(f"Episode finished! Total reward: {total_reward}")

Episodes:   0%|          | 0/1000 [00:00<?, ?it/s]

Episode finished! Total reward: -1528.0
Episode finished! Total reward: -1376.0
Episode finished! Total reward: -495.0
Episode finished! Total reward: -653.0
Episode finished! Total reward: -102.0
Episode finished! Total reward: -658.0
Episode finished! Total reward: -89.0
Episode finished! Total reward: -152.0
Episode finished! Total reward: -218.0
Episode finished! Total reward: -342.0
Episode finished! Total reward: -25.0
Episode finished! Total reward: -698.0
Episode finished! Total reward: -78.0
Episode finished! Total reward: -130.0
Episode finished! Total reward: -82.0
Episode finished! Total reward: -308.0
Episode finished! Total reward: -223.0
Episode finished! Total reward: -170.0
Episode finished! Total reward: -74.0
Episode finished! Total reward: -653.0
Episode finished! Total reward: -78.0
Episode finished! Total reward: -185.0
Episode finished! Total reward: -66.0
Episode finished! Total reward: -42.0
Episode finished! Total reward: -77.0
Episode finished! Total reward: 

In [10]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
  y=total_rewards,
  mode='lines',
  name='Raw',
  line=dict(color='steelblue')
))

fig.update_layout(
  title='Training Rewards',
  xaxis_title='Episode',
  yaxis_title='Total Reward',
)

fig.show()

In [14]:
# eval run

observation, info = environ.reset()

done = False

rewards = 0

while not done:
  print(environ.render())
  
  # choose an action
  action = select_action(q_table, observation, params.epsilon, n_actions, environ)

  # take the action
  new_observation, reward, terminated, truncated, info = environ.step(action)
  observation = new_observation

  rewards += float(reward)
  done = terminated or truncated

print(f"Total rewards: {rewards}")

o  o  o  o  o  o  o  o  o  o  o  o
o  o  o  o  o  o  o  o  o  o  o  o
o  o  o  o  o  o  o  o  o  o  o  o
x  C  C  C  C  C  C  C  C  C  C  T


o  o  o  o  o  o  o  o  o  o  o  o
o  o  o  o  o  o  o  o  o  o  o  o
x  o  o  o  o  o  o  o  o  o  o  o
o  C  C  C  C  C  C  C  C  C  C  T


o  o  o  o  o  o  o  o  o  o  o  o
x  o  o  o  o  o  o  o  o  o  o  o
o  o  o  o  o  o  o  o  o  o  o  o
o  C  C  C  C  C  C  C  C  C  C  T


o  o  o  o  o  o  o  o  o  o  o  o
o  x  o  o  o  o  o  o  o  o  o  o
o  o  o  o  o  o  o  o  o  o  o  o
o  C  C  C  C  C  C  C  C  C  C  T


o  o  o  o  o  o  o  o  o  o  o  o
o  o  x  o  o  o  o  o  o  o  o  o
o  o  o  o  o  o  o  o  o  o  o  o
o  C  C  C  C  C  C  C  C  C  C  T


o  o  o  o  o  o  o  o  o  o  o  o
o  o  o  x  o  o  o  o  o  o  o  o
o  o  o  o  o  o  o  o  o  o  o  o
o  C  C  C  C  C  C  C  C  C  C  T


o  o  o  o  o  o  o  o  o  o  o  o
o  o  x  o  o  o  o  o  o  o  o  o
o  o  o  o  o  o  o  o  o  o  o  o
o  C  C  C  C  C  C  C  C  C  C  T


o  o  

In [12]:
environ.close()